# Projet NoSQL & Big Data - Films TMDB (Cassandra + Redis)
## 04. Scripts d'administration

Dernière brique technique demandée par l'énoncé : dump, restauration, import
de fichiers volumineux, amélioration des performances, sécurisation. Comme
pour le notebook précédent, celui-ci est indépendant, je refais la connexion.


In [8]:
import os
import csv
import time
from dotenv import load_dotenv

load_dotenv()


True

In [9]:
from cassandra.cluster import Cluster, ProtocolVersion
from cassandra.auth import PlainTextAuthProvider

cloud_config = {"secure_connect_bundle": os.environ["ASTRA_DB_BUNDLE_PATH"], "connect_timeout": 30}
auth_provider = PlainTextAuthProvider(username="token", password=os.environ["ASTRA_DB_APPLICATION_TOKEN"])
cluster = Cluster(cloud=cloud_config, auth_provider=auth_provider, protocol_version=ProtocolVersion.V4)
session = cluster.connect()
session.set_keyspace("tmdb_platform")
print("Connectée à Cassandra.")


Connectée à Cassandra.


In [10]:
import redis

redis_client = redis.Redis(
    host=os.environ["REDIS_HOST"],
    port=int(os.environ["REDIS_PORT"]),
    password=os.environ["REDIS_PASSWORD"],
    username=os.environ.get("REDIS_USERNAME", "default"),
    decode_responses=True,
)
print("Connectée à Redis :", redis_client.ping())


Connectée à Redis : True


## 1. Dump (sauvegarde) - Cassandra

Astra DB ne propose pas d'outil `nodetool snapshot` accessible directement
(c'est un service géré, l'administration bas niveau du cluster physique
n'est pas exposée). La méthode adaptée à mon cas est d'exporter le contenu
de chaque table vers un CSV via une requête `SELECT *` complète - une
sauvegarde logique plutôt qu'une sauvegarde physique du cluster.

**Limite à noter** : sur un vrai volume de production (des millions de
lignes), un `SELECT *` complet sans pagination deviendrait problématique
(risque de timeout, consommation mémoire). Pour mes ~50 000 lignes réparties
sur 5 tables, ça reste raisonnable, mais ce n'est pas la méthode qu'on
utiliserait à plus grande échelle (Astra propose alors un vrai outil de
backup géré côté plateforme, hors de portée du tier gratuit).


In [11]:
def dump_table_to_csv(session, table_name, output_dir="backups"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, f"{table_name}.csv")

    rows = session.execute(f"SELECT * FROM {table_name}")
    rows = list(rows)
    if not rows:
        print(f"{table_name} : table vide, rien à exporter.")
        return output_path

    fieldnames = rows[0]._fields
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(fieldnames)
        for row in rows:
            writer.writerow(row)

    print(f"{table_name} : {len(rows)} lignes exportées vers {output_path}")
    return output_path


TABLES = ["movies_by_id", "movies_by_genre", "movies_by_director", "movies_by_actor", "movies_by_year"]

t0 = time.time()
for table in TABLES:
    dump_table_to_csv(session, table)
print(f"\nDump complet en {time.time() - t0:.1f}s")


movies_by_id : 4803 lignes exportées vers backups\movies_by_id.csv
movies_by_genre : 12144 lignes exportées vers backups\movies_by_genre.csv
movies_by_director : 4767 lignes exportées vers backups\movies_by_director.csv
movies_by_actor : 23589 lignes exportées vers backups\movies_by_actor.csv
movies_by_year : 4794 lignes exportées vers backups\movies_by_year.csv

Dump complet en 7.6s


## 2. Restauration - Cassandra

**Premier essai (abandonné)** : j'avais d'abord réinséré 20 lignes déjà
existantes par-dessus elles-mêmes, en pensant que ça prouvait la
restauration. En relisant mon propre code, j'ai réalisé que ça ne prouvait
rien du tout - la table contenait déjà ces lignes, donc le résultat aurait
été identique même si ma fonction de restauration ne faisait rien.

**Le vrai test** : pour qu'une restauration prouve quelque chose, il faut
d'abord que la donnée soit réellement absente. Je fais donc : je choisis 20
films réels, je les supprime, je vérifie que le compte a bien baissé de 20,
je les restaure depuis mon dump CSV, puis je vérifie que le compte est
revenu à la normale. C'est le seul enchaînement qui prouve vraiment que la
restauration fonctionne.


In [12]:
def parse_list_field(value):
    # les colonnes list<text> sont écrites par Cassandra comme "['a', 'b']" dans le CSV
    if not value or value == "None":
        return []
    return [v.strip(" '\"") for v in value.strip("[]").split(",") if v.strip()]

def row_to_params(row):
    def none_if_empty(v):
        return None if v in ("", "None") else v
    return (
        int(row["movie_id"]), row["title"], none_if_empty(row["director"]),
        parse_list_field(row["main_cast"]), parse_list_field(row["genres"]),
        int(row["release_year"]) if row["release_year"] not in ("", "None") else None,
        none_if_empty(row["release_date"]),
        float(row["runtime"]) if row["runtime"] not in ("", "None") else None,
        int(row["budget"]) if row["budget"] not in ("", "None") else None,
        int(row["revenue"]) if row["revenue"] not in ("", "None") else None,
        float(row["popularity"]), float(row["vote_average"]), int(row["vote_count"]),
        float(row["weighted_rating"]) if row["weighted_rating"] not in ("", "None") else None,
        row["original_language"], row["status"],
    )

insert_by_id_restore = session.prepare("""
    INSERT INTO movies_by_id
    (movie_id, title, director, main_cast, genres, release_year, release_date,
     runtime, budget, revenue, popularity, vote_average, vote_count,
     weighted_rating, original_language, status)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

print("Fonctions de restauration prêtes.")


Fonctions de restauration prêtes.


**Limite connue de `parse_list_field`** : le parsing des colonnes `list<text>`
depuis le CSV est volontairement simple (découpage sur les virgules et
suppression des guillemets). Ça fonctionne pour mon cas de test, mais
casserait sur un nom contenant lui-même une virgule ou une apostrophe (ex:
acteur "O'Brien"). Pour un vrai outil de restauration en production,
j'utiliserais plutôt un export en JSON (`json.dumps`/`json.loads`) qui gère
nativement les listes, plutôt que du CSV fait main pour des colonnes non
scalaires - je le note comme amélioration plutôt que de laisser croire que
ce dump CSV est robuste à toutes les données.


In [13]:
# Étape 1 : je choisis 20 films réels et je note leur movie_id
test_ids = [r.movie_id for r in session.execute("SELECT movie_id FROM movies_by_id LIMIT 20")]
count_avant = session.execute("SELECT COUNT(*) FROM movies_by_id").one().count
print(f"Avant suppression : {count_avant} lignes dans movies_by_id")
print(f"20 movie_id choisis pour le test : {test_ids}")


Avant suppression : 4803 lignes dans movies_by_id
20 movie_id choisis pour le test : [1584, 13909, 16096, 9067, 136797, 24066, 37958, 43610, 11194, 82695, 769, 18045, 9495, 7863, 2453, 14799, 80271, 86304, 46989, 14112]


In [14]:
# Étape 2 : je supprime réellement ces 20 lignes
for mid in test_ids:
    session.execute("DELETE FROM movies_by_id WHERE movie_id = %s", (mid,))

count_apres_suppression = session.execute("SELECT COUNT(*) FROM movies_by_id").one().count
print(f"Après suppression : {count_apres_suppression} lignes "
      f"(attendu : {count_avant} - 20 = {count_avant - 20})")
assert count_apres_suppression == count_avant - 20, "La suppression n'a pas le compte attendu !"


Après suppression : 4783 lignes (attendu : 4803 - 20 = 4783)


In [15]:
# Étape 3 : je restaure UNIQUEMENT ces 20 films, depuis le dump CSV, en filtrant sur test_ids
test_ids_set = set(test_ids)
restored = 0

with open("backups/movies_by_id.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if int(row["movie_id"]) in test_ids_set:
            session.execute(insert_by_id_restore, row_to_params(row))
            restored += 1

print(f"{restored} lignes restaurées depuis le dump (sur les {len(test_ids)} attendues).")


20 lignes restaurées depuis le dump (sur les 20 attendues).


In [16]:
# Étape 4 : vérification finale - le compte doit être revenu à la normale
count_apres_restauration = session.execute("SELECT COUNT(*) FROM movies_by_id").one().count
print(f"Après restauration : {count_apres_restauration} lignes (attendu : {count_avant})")

if count_apres_restauration == count_avant:
    print("Restauration validée : le compte est revenu exactement au niveau initial.")
else:
    print("ATTENTION : le compte ne correspond pas, la restauration a un problème.")


Après restauration : 4803 lignes (attendu : 4803)
Restauration validée : le compte est revenu exactement au niveau initial.


## 3. Import de fichiers volumineux

C'est un point qu'on a déjà réellement résolu dans le notebook 02, mais à
préciser honnêtement : ma première version de l'ETL (insertion séquentielle,
une requête à la fois) n'est jamais allée au bout d'un run complet - je suis
passée à `execute_concurrent_with_args` avant d'avoir un temps total mesuré
pour la version séquentielle. Je ne peux donc pas affirmer un chiffre mesuré
pour elle ; j'**estime**, sur la base du débit typique d'une requête
synchrone vers Astra (latence réseau aller-retour), qu'un import séquentiel
complet des ~50 000 lignes aurait pris entre 25 et 90 minutes.

Ce qui est en revanche réellement **mesuré**, c'est le temps obtenu avec
`execute_concurrent_with_args` dans le notebook 02 : ~50 000 lignes en un
peu plus d'une minute.

Je documente ici la fonction générique réutilisable, plutôt que de la
dupliquer telle qu'utilisée dans le notebook 02.


In [ ]:
from cassandra.concurrent import execute_concurrent_with_args

def bulk_import(session, prepared_stmt, params_list, concurrency=100, label=""):
    t0 = time.time()
    results = execute_concurrent_with_args(session, prepared_stmt, params_list, concurrency=concurrency)
    errors = [(i, r) for i, (success, r) in enumerate(results) if not success]
    elapsed = time.time() - t0
    print(f"{label} : {len(params_list)} lignes en {elapsed:.1f}s ({len(errors)} erreurs, concurrency={concurrency})")
    return errors

print("Fonction bulk_import prête (déjà utilisée et mesurée dans le notebook 02).")
print("Temps mesuré avec execute_concurrent_with_args : ~50 000 lignes en 1min28.")
print("Temps séquentiel : estimé entre 25 et 90 min (jamais mesuré jusqu'au bout, voir section 3 ci-dessus).")


## 4. Amélioration des performances - récapitulatif

Trois optimisations réellement appliquées dans ce projet :

1. **Requêtes préparées** (`session.prepare`) plutôt que des requêtes brutes
   reconstruites à chaque insertion - le driver compile la requête une
   seule fois.
2. **Exécution concurrente** (`execute_concurrent_with_args`) plutôt que
   séquentielle - gain **estimé** x30 à x60 sur l'ETL (le temps concurrent
   est mesuré, le temps séquentiel est une estimation puisqu'il n'a jamais
   été mesuré jusqu'au bout - voir section 3).
3. **Filtrage par table selon sa propre clé primaire** plutôt qu'un filtre
   unique trop restrictif appliqué à toutes les tables - ça évite d'exclure
   des lignes exploitables (30 films sans réalisateur qui restaient valides
   pour `movies_by_genre`, par exemple).

Un point identifié mais non appliqué, à mentionner honnêtement : le `COUNT(*)`
que j'utilise pour vérifier mes volumes scanne toute la table à chaque appel -
acceptable ponctuellement pour vérifier une insertion, mais à proscrire en
boucle ou en usage fréquent sur une vraie table de production.


## 5. Sécurisation

Récapitulatif de ce qui est déjà en place dans ce projet, plus un point que
je n'ai pas encore fait et que je documente comme amélioration possible.

**Déjà en place :**
- Aucun identifiant en clair dans le code : tout passe par `.env`, exclu du
  repo GitHub via `.gitignore` (`.env`, `secure-connect-*.zip`).
- Connexion chiffrée en TLS de bout en bout : le secure connect bundle
  Astra embarque les certificats nécessaires, Redis Cloud propose le TLS
  sur son tier gratuit.
- Le token Astra a été généré avec le rôle "Database Administrator" - ce
  qui est **excessif** pour une simple application de lecture/écriture sur
  mes tables.

**Amélioration identifiée, pas appliquée (par manque de temps, à
documenter honnêtement plutôt qu'à cacher)** : Astra permet de créer des
tokens avec des rôles plus restreints ("Read Only", ou un rôle personnalisé
limité à un keyspace précis). Pour une vraie mise en production, je
séparerais le token utilisé par le notebook d'exploration (accès large,
utilisé une fois) du token utilisé par une éventuelle WebApp en lecture
seule (accès restreint), plutôt que de réutiliser le même token
"Database Administrator" partout comme je le fais actuellement.


## 6. Dump / restauration / sécurité côté Redis

Redis fonctionne différemment de Cassandra sur ce point : la persistance
n'est pas gérée par un export manuel mais par des snapshots natifs
(`RDB`) que Redis Cloud gère automatiquement en arrière-plan.


In [18]:
# Déclenchement manuel d'une sauvegarde avant une opération risquée
try:
    redis_client.bgsave()
    print("Sauvegarde Redis (BGSAVE) déclenchée en arrière-plan.")
except Exception as e:
    print("BGSAVE non autorisé sur ce tier (normal sur certains plans gratuits) :", e)


BGSAVE non autorisé sur ce tier (normal sur certains plans gratuits) : command 'bgsave' is not allowed


**Résultat obtenu en pratique** : `command 'bgsave' is not allowed` - mon
tier gratuit Redis Cloud bloque effectivement le déclenchement manuel d'un
snapshot. Ce n'est pas un bug de mon code, c'est une restriction imposée
par le plan gratuit : sur ce tier, Redis Cloud gère les sauvegardes
automatiquement en arrière-plan, sans laisser le client en déclencher une à
la demande. Je le documente comme une vraie limite de mon environnement,
pas comme un problème résolu à moitié.

**Sécurité Redis déjà en place** : mot de passe obligatoire
(`REDIS_PASSWORD`), connexion via un compte nommé plutôt qu'anonyme
(`REDIS_USERNAME=default`), et comme pour Astra, aucun identifiant en clair
dans le code.


## Bilan

- Dump Cassandra : fonctionnel, les 5 tables exportées (~50 000 lignes au
  total) en moins de 8 secondes.
- Restauration Cassandra : testée avec un vrai cycle supprimer → vérifier →
  restaurer → revérifier sur 20 films réels, pas une réinsertion sur des
  données déjà présentes (erreur de méthode corrigée après une première
  relecture de mon propre code).
- Import volumineux : déjà résolu dans le notebook 02, avec un temps mesuré
  pour la version concurrente (~50 000 lignes en 1min28) et un gain de
  x30-60 estimé par rapport à la version séquentielle (jamais mesurée
  jusqu'au bout, à ne pas présenter comme un chiffre mesuré).
- Performances : 3 optimisations réellement appliquées, documentées avec
  leur justification.
- Sécurité : bonnes pratiques appliquées (`.env`, TLS), une limite identifiée
  et assumée (token trop permissif, pas de séparation des rôles d'accès).
- Redis : `BGSAVE` manuel refusé par le tier gratuit (vérifié en conditions
  réelles, pas supposé) - sauvegarde automatique gérée par la plateforme.

Prochaine étape : la WebApp de test, le README, et les slides de présentation.
